# 01 - Why Geometric Algebra?

This notebook is a standalone, executable version of the opening motivation for
*Geometric Algebra for Computer Science*. You should be able to read it without
keeping the book open: each construction is introduced, translated into the
coordinate code used here, visualized, and checked with numerical invariants.

The theme is object-oriented geometry in the mathematical sense. Instead of treating
a line, a circle, a plane, and a transformation as unrelated data structures, geometric
algebra aims to represent them as elements of one algebra. Operations then act on the
whole object, not on a pile of special-case coordinates.


## The chapter idea in one picture

The first chapter starts with a deliberately concrete scene:

- choose three points and regard them as defining a circle;
- choose two points and regard them as defining a line;
- rotate the circle around the line;
- choose a plane and reflect the line, circle, and rotated circle in it;
- replace the plane by a sphere to show that the same operator-shaped idea can create a different transformation.

In geometric algebra notation, the impressive part is that these actions have a common
shape. A transformation is usually written as an object wrapped around the thing being
transformed, for example $X \mapsto R X R^{-1}$ for a rotation-like operator or
$X \mapsto -P X P^{-1}$ for a reflection-like operator. This wrapping pattern is often
called a sandwich product.

This notebook does not implement the geometric product yet. Instead, it uses ordinary
NumPy coordinate formulas that produce the same visible effects for this chapter's
examples. Think of the code as a coordinate shadow of the algebra: faithful enough to
build intuition, intentionally simpler than the full machinery.


## Translation guide

The table below names the geometric objects in plain language and explains how this
notebook represents them.

| Geometric idea | Full geometric-algebra style | Coordinate stand-in used here |
|---|---|---|
| Point | A modeled point element, especially in the conformal model | A NumPy array `[x, y, z]` |
| Circle through three points | Build one object from three point elements | Fit the unique Euclidean circle through `c1`, `c2`, and `c3` |
| Line through two points | Build one object from two points plus the point at infinity | Store one point and one unit direction |
| Plane from point and normal | Use the dual plane as a reflector | Store one point and one unit normal |
| Rotation around a line | Use a rotor and sandwich the circle with it | Use a 3-D rotation matrix around the line direction |
| Reflection in a plane | Use the plane as an operator on any object | Reflect sampled points and reconstruct the line |
| Reflection in a sphere | Use a sphere operator with the same algebraic pattern | Apply classical sphere inversion to sampled points |

The important habit is to read every variable as a geometric entity first and as
coordinates second. The code starts with coordinates only because a computer needs
numbers before it can draw a picture.


## Notebook route

1. Import reusable helpers and set the artifact directory.
2. Build the scene from a few coordinate handles.
3. Draw the circle, line, rotation, plane reflection, and interpolation steps.
4. Replace the plane by a sphere and observe how inversion changes the objects.
5. Verify invariants so the notebook checks the geometry, not just the plots.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import plotly.graph_objects as go

# Find the project root even if the notebook is opened from this chapter folder.
PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "utils" / "artifacts.py").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.artifacts import save_plotly_html
from utils.chapter01_geometry import (
    COLORS,
    Circle3D,
    Line3D,
    Plane3D,
    Sphere3D,
    curve_trace,
    finish_figure,
    line_trace,
    plane_surface_trace,
    point_trace,
    rotate_points_about_line,
    sphere_surface_trace,
)

np.set_printoptions(precision=4, suppress=True)
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
PROJECT_ROOT


WindowsPath('D:/Geometry')

## Shared geometry helpers

The reusable coordinate geometry lives in `utils/chapter01_geometry.py`. Keeping that
code outside the notebook makes the story cells easier to read: this page can focus on
the geometric construction while the helper module owns details such as normalizing
vectors, sampling circles, building Plotly traces, and saving figures.

The most useful classes are:

- `Circle3D`, which constructs a circle from three non-collinear points and samples it;
- `Line3D`, which stores a point and a unit direction;
- `Plane3D`, which computes signed distances and reflections;
- `Sphere3D`, which computes sphere inversion.


## A circle, a line, and a reflecting plane

The scene starts from seven small pieces of input data. Five are point handles:
`c1`, `c2`, and `c3` determine the circle, while `l1` and `l2` determine the line.
The plane is determined by one point on the plane and one normal vector.

In the full algebra, these inputs would be combined with products that create complete
geometric objects. Here the helper classes perform the equivalent coordinate work:
`Circle3D.through_three_points(c1, c2, c3)` fits the circle, `Line3D.through(l1, l2)`
builds the line, and `Plane3D(plane_point, plane_normal)` builds the reflector.


In [2]:
# Scene inputs: these are the coordinates that replace the book's interactive handles.
c1 = np.array([1.10, 0.95, 0.18])
c2 = np.array([-0.82, 0.62, -0.10])
c3 = np.array([0.20, -0.92, 0.55])
l1 = np.array([-1.25, -0.90, -0.55])
l2 = np.array([1.20, 0.70, 0.82])
plane_point = np.array([0.08, -0.12, 0.05])
plane_normal = np.array([0.70, -0.38, 0.61])

C = Circle3D.through_three_points(c1, c2, c3)
L = Line3D.through(l1, l2)
plane = Plane3D(plane_point, plane_normal)

circle_points = C.sample(180)
phi = np.pi / 2.0
rotated_circle_points = rotate_points_about_line(circle_points, L, phi)
reflected_line = plane.reflect_line(L)
reflected_circle_points = plane.reflect_points(circle_points)
reflected_rotated_points = plane.reflect_points(rotated_circle_points)

print(f"circle center: {C.center}, radius: {C.radius:.3f}")
print(f"line direction: {L.direction}")
print(f"plane normal: {plane.normal}")


circle center: [0.2138 0.1974 0.2262], radius: 1.164
line direction: [0.7583 0.4952 0.424 ]
plane normal: [ 0.6977 -0.3788  0.608 ]


## Figure 1: object-level transformations

In the plot below, the red line is the rotation axis and the green circle is the original
circle through `c1`, `c2`, and `c3`. The blue circle is the result of rotating the green
circle by `phi = pi / 2` around the red line. The pale green curves show intermediate
rotation steps, which make the motion easier to read.

The yellow sheet is the reflecting plane. The purple and pale blue curves are the same
objects after plane reflection. The main lesson is not the particular coordinate values;
it is that a transformation can be applied to a whole line or circle as a geometric
object. Geometric algebra is designed to make that object-level view the natural one.


In [3]:
fig_plane = go.Figure()
fig_plane.add_trace(plane_surface_trace(plane, "reflecting plane"))
fig_plane.add_trace(line_trace(L, "L", COLORS["line"]))
fig_plane.add_trace(line_trace(reflected_line, "reflected L", COLORS["reflected"], width=5))
fig_plane.add_trace(curve_trace(circle_points, "C through c1,c2,c3", COLORS["circle"], width=8))
fig_plane.add_trace(curve_trace(rotated_circle_points, "R C / R", COLORS["rotated"], width=8))
fig_plane.add_trace(curve_trace(reflected_circle_points, "plane reflection of C", COLORS["reflected"], width=6))
fig_plane.add_trace(
    curve_trace(reflected_rotated_points, "plane reflection of rotated C", "#17becf", width=6)
)

for alpha in np.linspace(0.1, 0.9, 9):
    interpolated = rotate_points_about_line(circle_points, L, alpha * phi)
    fig_plane.add_trace(
        curve_trace(interpolated, f"interpolation {alpha:.1f}", COLORS["light_green"], width=3)
    )
    fig_plane.add_trace(
        curve_trace(
            plane.reflect_points(interpolated),
            f"reflected interpolation {alpha:.1f}",
            COLORS["light_blue"],
            width=3,
        )
    )

fig_plane.add_trace(point_trace(np.array([c1, c2, c3, l1, l2]), ["c1", "c2", "c3", "l1", "l2"], "#111111"))
finish_figure(fig_plane, "Chapter 1 scene: circle, line rotation, and plane reflection")

plane_path = save_plotly_html(
    fig_plane,
    "chapter-01",
    "plots",
    "line-circle-plane-reflection.html",
    root=ARTIFACT_ROOT,
)
print(f"saved {plane_path}")
fig_plane


saved D:\Geometry\artifacts\chapter-01\plots\line-circle-plane-reflection.html


## What the first figure demonstrates

There are three linked ideas in this scene.

First, construction is geometric. A circle is specified by points on it, not by a center,
radius, and hand-written basis vectors. The helper computes those derived quantities.

Second, transformations preserve structure. A rotation around `L` keeps every point of
the circle at the same distance from `L`; a plane reflection reverses signed distance to
the plane; both operations turn a circle into another circle.

Third, interpolation belongs to the operator. The pale curves are not separate circles
invented by a drawing routine. They are generated by applying partial rotations to the
same original circle. Later, true rotors will make this interpolation algebraic instead
of matrix-based.


## Sphere inversion variant

The chapter then swaps the plane reflector for a sphere reflector. This is a useful
stress test for the notation: the operator-shaped idea is the same, but the geometry is
different. Plane reflection sends lines to lines and circles to circles. Sphere inversion
can bend a line into a circle-like curve when the line does not pass through the sphere
center, and it changes distances by a reciprocal rule.

In coordinate terms, sphere inversion is simple. For a sphere with center `c` and radius
`r`, a point `x` moves along the ray from `c` through `x` so that

$$
\|x-c\| \; \|x'-c\| = r^2
$$

Points outside the sphere move inside, points inside move outside, and points on the
sphere stay fixed. That reciprocal distance rule is what the next code cell applies to
the sampled line and circle points.


In [4]:
sphere = Sphere3D(center=np.array([0.62, -0.18, 0.12]), radius=1.12)
line_points = L.sample(-3.0, 3.0, 220)
inverted_line_points = sphere.invert_points(line_points)
inverted_circle_points = sphere.invert_points(circle_points)
inverted_rotated_points = sphere.invert_points(rotated_circle_points)

fig_sphere = go.Figure()
fig_sphere.add_trace(sphere_surface_trace(sphere, "inversion sphere"))
fig_sphere.add_trace(line_trace(L, "L", COLORS["line"], t_min=-3.0, t_max=3.0))
fig_sphere.add_trace(curve_trace(circle_points, "C", COLORS["circle"], width=8))
fig_sphere.add_trace(curve_trace(rotated_circle_points, "R C / R", COLORS["rotated"], width=7))
fig_sphere.add_trace(curve_trace(inverted_line_points, "sphere inversion of L", COLORS["reflected"], width=6))
fig_sphere.add_trace(curve_trace(inverted_circle_points, "sphere inversion of C", "#ff7f0e", width=6))
fig_sphere.add_trace(curve_trace(inverted_rotated_points, "sphere inversion of rotated C", "#17becf", width=6))
fig_sphere.add_trace(
    point_trace(np.array([sphere.center, c1, c2, c3]), ["center", "c1", "c2", "c3"], "#111111")
)
finish_figure(fig_sphere, "Chapter 1 variant: replacing the plane with a sphere inversion")

sphere_path = save_plotly_html(
    fig_sphere,
    "chapter-01",
    "plots",
    "sphere-inversion-variant.html",
    root=ARTIFACT_ROOT,
)
print(f"saved {sphere_path}")
fig_sphere


saved D:\Geometry\artifacts\chapter-01\plots\sphere-inversion-variant.html


## What the sphere variant demonstrates

The sphere plot is intentionally parallel to the plane plot. We keep the original line,
circle, and rotated circle, then change only the reflector. This mirrors the programming
advantage promised by geometric algebra: once the objects and operators live in a
common system, replacing one geometric operator can update a whole construction.

Our coordinate implementation still has special-case functions: `Plane3D.reflect_points`
for a plane and `Sphere3D.invert_points` for a sphere. Later notebooks can replace
those separate formulas with the common product language that this chapter is pointing
toward.


## Sanity checks

A self-contained computational notebook should check the claims it makes. These tests
verify geometric invariants:

- reflecting in a plane flips signed distance to that plane;
- rotating around a line preserves distance to the line;
- the rotation interpolation starts and ends at the expected circles;
- sphere inversion satisfies the radius-distance product rule;
- the HTML artifacts were written to disk.

The printed errors should be near floating-point zero.


In [5]:
# Plane reflection changes the sign of signed distance to the plane.
original_distances = plane.signed_distance(circle_points)
reflected_distances = plane.signed_distance(reflected_circle_points)
reflection_error = np.max(np.abs(reflected_distances + original_distances))
assert np.allclose(reflected_distances, -original_distances, atol=1e-9)

# Rotation about a line preserves every point's distance to that line.
original_axis_distances = L.distance_to_points(circle_points)
rotated_axis_distances = L.distance_to_points(rotated_circle_points)
rotation_error = np.max(np.abs(rotated_axis_distances - original_axis_distances))
assert np.allclose(rotated_axis_distances, original_axis_distances, atol=1e-9)

# Interpolation endpoints match the original and final rotated samples.
start_error = np.max(np.abs(rotate_points_about_line(circle_points, L, 0.0) - circle_points))
end_error = np.max(np.abs(rotate_points_about_line(circle_points, L, phi) - rotated_circle_points))
assert np.allclose(rotate_points_about_line(circle_points, L, 0.0), circle_points, atol=1e-9)
assert np.allclose(rotate_points_about_line(circle_points, L, phi), rotated_circle_points, atol=1e-9)

# Sphere inversion has the expected radius-distance product.
line_distance_before = np.linalg.norm(line_points - sphere.center, axis=1)
line_distance_after = np.linalg.norm(inverted_line_points - sphere.center, axis=1)
inversion_error = np.max(np.abs(line_distance_before * line_distance_after - sphere.radius**2))
assert np.allclose(line_distance_before * line_distance_after, sphere.radius**2, atol=1e-9)

assert Path(plane_path).exists()
assert Path(sphere_path).exists()

print("sanity checks passed")
print(f"plane reflection signed-distance error: {reflection_error:.2e}")
print(f"rotation axis-distance error: {rotation_error:.2e}")
print(f"rotation interpolation start/end errors: {start_error:.2e}, {end_error:.2e}")
print(f"sphere inversion radius-product error: {inversion_error:.2e}")
print(f"artifacts: {plane_path}, {sphere_path}")


sanity checks passed
plane reflection signed-distance error: 6.66e-16
rotation axis-distance error: 5.55e-16
rotation interpolation start/end errors: 2.22e-16, 0.00e+00
sphere inversion radius-product error: 4.44e-16
artifacts: D:\Geometry\artifacts\chapter-01\plots\line-circle-plane-reflection.html, D:\Geometry\artifacts\chapter-01\plots\sphere-inversion-variant.html


## Chapter takeaways

This notebook has used coordinate formulas, but the conceptual target is broader.
Geometric algebra treats subspaces, geometric objects, and many transformations as
elements of one algebraic system. The same product family can build objects, measure
relationships, take complements, and express transformations.

For programming, the promise is practical: fewer one-off representations, fewer
conversions between line/circle/plane formats, and transformations that can be written
at the level of the geometry being modeled. The price is learning the algebraic products
carefully. The next notebooks should therefore move from this visual motivation into
the core products: the outer product for spanning oriented subspaces, metric products
for measuring and projecting, duality for complements, and the geometric product for
the unified operator language.
